# Predicción de enfermedad cardíaca: XGBoost y entrega a Kaggle

Tercer modelo de la serie. Al final se entrena el modelo con todos los datos y se genera el archivo de entrega para la competencia de Kaggle.

**Dataset:** registros de pacientes con 11 variables clínicas y la columna objetivo `HeartDisease`
(1 = tiene enfermedad cardíaca, 0 = no tiene).

| Variable | Descripción |
|---|---|
| Age | Edad |
| Sex | Sexo (M / F) |
| ChestPainType | Tipo de dolor de pecho (ASY, ATA, NAP, TA) |
| RestingBP | Presión arterial sistólica en reposo (mm Hg) |
| Cholesterol | Colesterol (mg/dl). **Un valor 0 indica dato faltante** |
| FastingBS | Glucemia en ayunas > 120 mg/dl (1 / 0) |
| RestingECG | Electrocardiograma en reposo (Normal, ST, LVH) |
| MaxHR | Frecuencia cardíaca máxima |
| ExerciseAngina | Angina inducida por ejercicio (Y / N) |
| Oldpeak | Depresión del segmento ST |
| ST_Slope | Pendiente del segmento ST (Up, Flat, Down) |

La métrica pedida por la consigna es **accuracy**.

## Notas de metodología

- XGBoost maneja **valores faltantes** de forma nativa, así que los colesterol en 0 se reemplazan por `NaN`
  y no hace falta imputar.
- Las variables categóricas se usan directamente con `enable_categorical=True`. Para eso se convierten
  a tipo `category` con **la misma lista de categorías** en train, test y en los datos de Kaggle,
  así cada categoría se codifica igual en todos los conjuntos.
- El umbral de decisión se elige con validación cruzada sobre train, no mirando el test.

In [ ]:
!pip install -q -U xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (train_test_split, GridSearchCV, RandomizedSearchCV,
                                     StratifiedKFold, cross_val_predict)
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, auc)
from xgboost import XGBClassifier

In [ ]:
# Descarga de los datos desde Google Drive (compartidos como "Cualquier persona con el enlace")
import os
import gdown

ARCHIVOS = {
"heart_train.csv": "10zgipXXcGiJn0xqGz90JwtVFOgcdl6UO",
"heart_test.csv": "1QUbRxSkwp3_noQZXZPKwLx1w9mnQzzT0",
"sample_submission.csv": "11PsQb0fXClYkohAig9EzE4ZyTu5hITwy",
}

for nombre, file_id in ARCHIVOS.items():
    if not os.path.exists(nombre):
        gdown.download(id=file_id, output=nombre, quiet=True)

In [ ]:
df = pd.read_csv("heart_train.csv")

CATEGORICAS = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]

# Lista fija de categorías: así cada valor recibe el mismo código en todos los conjuntos
TIPOS_CATEGORICOS = {col: pd.CategoricalDtype(sorted(df[col].unique())) for col in CATEGORICAS}


def preparar(X):
    X = X.copy()
    for col, tipo in TIPOS_CATEGORICOS.items():
        X[col] = X[col].astype(tipo)
    X["Cholesterol"] = X["Cholesterol"].replace(0, np.nan)   # 0 = dato faltante
    return X


X = preparar(df.drop(columns="HeartDisease"))
y = df["HeartDisease"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=10, stratify=y
)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## Búsqueda de hiperparámetros con early stopping

Separo un 20% de train como conjunto de validación para el **early stopping**: cada modelo deja de agregar
árboles si en 20 rondas seguidas no mejora sobre la validación. El conjunto de test no participa.

In [ ]:
X_sub_train, X_val, y_sub_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

parametros = {
    "n_estimators": [100, 200, 300],           # Cantidad máxima de árboles
    "max_depth": [3, 4, 5, 6],                 # Profundidad máxima de cada árbol
    "learning_rate": [0.01, 0.05, 0.1, 0.2],   # Tasa de aprendizaje
    "subsample": [0.6, 0.8, 1.0],              # Proporción de filas usadas en cada árbol
    "colsample_bytree": [0.6, 0.8, 1.0],       # Proporción de columnas usadas en cada árbol
    "gamma": [0, 0.1, 0.2],                    # Mejora mínima de la pérdida para hacer un split (regularización)
}

modelo_base = XGBClassifier(
    enable_categorical=True,
    early_stopping_rounds=20,
    eval_metric="logloss",
    random_state=42,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

busqueda = RandomizedSearchCV(
    modelo_base,
    param_distributions=parametros,
    n_iter=100,
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)
busqueda.fit(X_sub_train, y_sub_train, eval_set=[(X_val, y_val)], verbose=False)

mejor_modelo = busqueda.best_estimator_
print(f"Mejores parámetros: {busqueda.best_params_}")
print(f"Mejor accuracy (CV): {busqueda.best_score_:.4f}")
print(f"Árboles usados tras early stopping: {mejor_modelo.best_iteration + 1}")

## Evaluación en test

In [ ]:
def evaluar(modelo, X, y, umbral=0.5, titulo=""):
    """Imprime las métricas en test y grafica la matriz de confusión."""
    probas = modelo.predict_proba(X)[:, 1]
    pred = (probas >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()

    print(f"{titulo} (umbral = {umbral:.2f})")
    print(f"  Accuracy:      {accuracy_score(y, pred):.4f}")
    print(f"  Sensibilidad:  {tp / (tp + fn):.4f}   (enfermos detectados)")
    print(f"  Especificidad: {tn / (tn + fp):.4f}   (sanos bien clasificados)")
    print(f"  ROC-AUC:       {roc_auc_score(y, probas):.4f}")
    return probas


def graficar_matriz_confusion(y, pred, titulo):
    plt.figure(figsize=(6, 5))
    sns.heatmap(confusion_matrix(y, pred), annot=True, fmt="d", cmap="Blues", vmin=0,
                xticklabels=["Sano", "Enfermo"], yticklabels=["Sano", "Enfermo"])
    plt.title(titulo)
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.show()


def graficar_roc(y, probas, titulo):
    fpr, tpr, _ = roc_curve(y, probas)
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"Curva ROC (AUC = {auc(fpr, tpr):.4f})")
    plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Clasificador aleatorio")
    plt.xlabel("Tasa de falsos positivos (FPR)")
    plt.ylabel("Tasa de verdaderos positivos (TPR)")
    plt.title(titulo)
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
print(f"Accuracy en train: {accuracy_score(y_sub_train, mejor_modelo.predict(X_sub_train)):.4f}\n")
probas_test = evaluar(mejor_modelo, X_test, y_test, titulo="Test")
print()
print(classification_report(y_test, mejor_modelo.predict(X_test), target_names=["Sano", "Enfermo"]))
graficar_matriz_confusion(y_test, mejor_modelo.predict(X_test), "Matriz de confusión: XGBoost")

In [ ]:
graficar_roc(y_test, probas_test, "Curva ROC: XGBoost")

## Importancia de las variables

In [ ]:
importancias = pd.Series(mejor_modelo.feature_importances_, index=X_train.columns).sort_values()

plt.figure(figsize=(9, 6))
importancias.plot(kind="barh")
plt.title("Importancia de las variables: XGBoost")
plt.tight_layout()
plt.show()

## ¿El umbral de decisión impacta en el accuracy?

Busco el umbral que maximiza el accuracy usando **probabilidades de validación cruzada sobre train**.
Elegirlo mirando el test haría que el resultado en test sea optimista.

In [ ]:
probas_cv = cross_val_predict(mejor_modelo, X_sub_train, y_sub_train, cv=cv, method="predict_proba", params={"eval_set": [(X_val, y_val)], "verbose": False})[:, 1]

umbrales = np.arange(0.05, 0.96, 0.01)
accuracies = [accuracy_score(y_sub_train, (probas_cv >= u).astype(int)) for u in umbrales]
umbral_optimo = umbrales[int(np.argmax(accuracies))]

plt.figure(figsize=(8, 5))
plt.plot(umbrales, accuracies)
plt.axvline(0.5, color="gray", linestyle="--", label="Umbral por defecto (0,5)")
plt.axvline(umbral_optimo, color="darkorange", linestyle="--", label=f"Umbral óptimo ({umbral_optimo:.2f})")
plt.xlabel("Umbral")
plt.ylabel("Accuracy (validación cruzada)")
plt.title("Accuracy según el umbral de decisión")
plt.legend()
plt.show()

print(f"Accuracy CV con umbral 0.50: {accuracy_score(y_sub_train, (probas_cv >= 0.5).astype(int)):.4f}")
print(f"Accuracy CV con umbral {umbral_optimo:.2f}: {max(accuracies):.4f}")

In [ ]:
_ = evaluar(mejor_modelo, X_test, y_test, umbral=0.5, titulo="Test")
print()
_ = evaluar(mejor_modelo, X_test, y_test, umbral=umbral_optimo, titulo="Test")

Si la mejora con el umbral óptimo es chica, conviene quedarse con 0,5: un umbral muy ajustado
a un dataset de este tamaño puede no generalizar.

# Entrega a Kaggle

Para la entrega final entreno el modelo con **todos** los datos de `heart_train.csv`, usando los mejores
hiperparámetros y la cantidad de árboles que encontró el early stopping. Las predicciones usan el umbral
elegido por validación cruzada.

In [ ]:
X_kaggle = preparar(pd.read_csv("heart_test.csv")[X.columns])
sample_submission = pd.read_csv("sample_submission.csv")

parametros_finales = {**busqueda.best_params_, "n_estimators": mejor_modelo.best_iteration + 1}
modelo_final = XGBClassifier(**parametros_finales, enable_categorical=True, eval_metric="logloss", random_state=42)
modelo_final.fit(X, y)

predicciones = (modelo_final.predict_proba(X_kaggle)[:, 1] >= umbral_optimo).astype(int)
assert len(predicciones) == len(sample_submission), "La cantidad de filas no coincide con sample_submission"

# Mismo formato que sample_submission: se reemplaza solo la columna de predicción
submission = sample_submission.copy()
submission[submission.columns[-1]] = predicciones
submission.to_csv("submission.csv", index=False)

print(f"Proporción de enfermos predicha: {predicciones.mean():.2%}")
submission.head()